# RGB Baseline Model

In this notebook, we train a model to detect violence in videos using RGB frames.

We:
- prepare the dataset
- load video clips
- train a 3D CNN (R3D-18)
- evaluate performance on validation data
- save trained models

This model will later be compared with pose-based approaches.

Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
from pathlib import Path

BASE = "/content/drive/MyDrive/violence-detection"
DATA_DIR = f"{BASE}/data"
RAW_DIR = f"{DATA_DIR}/raw"
PROCESSED_DIR = f"{DATA_DIR}/processed"

RGB_RAW_ZIP = f"{RAW_DIR}/RWF2000.zip"
RGB_CLEAN_DIR = f"{PROCESSED_DIR}/rgb/rwf2000_clean"

CHECKPOINT_DIR = f"{BASE}/checkpoints/rgb"
RESULTS_DIR = f"{BASE}/results"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Paths ready.")

Prepare cleaned RGB dataset

In [ ]:
import zipfile, shutil
from tqdm import tqdm

zip_path = RGB_RAW_ZIP
out_root = Path("/content/rwf2000_clean")

if out_root.exists():
    shutil.rmtree(out_root)

(out_root / "train" / "Fight").mkdir(parents=True, exist_ok=True)
(out_root / "train" / "NonFight").mkdir(parents=True, exist_ok=True)
(out_root / "val" / "Fight").mkdir(parents=True, exist_ok=True)
(out_root / "val" / "NonFight").mkdir(parents=True, exist_ok=True)

video_exts = (".avi", ".mp4", ".mov", ".mkv")
counts = {
    ("train", "Fight"): 0,
    ("train", "NonFight"): 0,
    ("val", "Fight"): 0,
    ("val", "NonFight"): 0
}

with zipfile.ZipFile(zip_path, "r") as z:
    members = [m for m in z.infolist() if not m.is_dir()]

    for m in tqdm(members, desc="Extracting safe RGB dataset"):
        name = m.filename

        if not name.lower().endswith(video_exts):
            continue

        split = None
        if "/train/" in name or name.startswith("train/") or "\\train\\" in name:
            split = "train"
        elif "/val/" in name or name.startswith("val/") or "\\val\\" in name:
            split = "val"
        else:
            continue

        cls = None
        if "NonFight" in name or "Nonfight" in name or "nonfight" in name:
            cls = "NonFight"
        elif "Fight" in name or "fight" in name:
            cls = "Fight"
        else:
            continue

        idx = counts[(split, cls)]
        safe_filename = f"{idx:06d}.avi"
        out_path = out_root / split / cls / safe_filename

        with z.open(m, "r") as src, open(out_path, "wb") as dst:
            shutil.copyfileobj(src, dst)

        counts[(split, cls)] += 1

print("Done:", counts)

Save cleaned RGB dataset to Drive

In [ ]:
import shutil
import os

if os.path.exists(RGB_CLEAN_DIR):
    shutil.rmtree(RGB_CLEAN_DIR)

shutil.copytree("/content/rwf2000_clean", RGB_CLEAN_DIR)
print("Copied cleaned RGB dataset to Drive:", RGB_CLEAN_DIR)

Sanity checks

In [ ]:
import os

base = RGB_CLEAN_DIR
for split in ["train", "val"]:
    for cls in ["Fight", "NonFight"]:
        p = os.path.join(base, split, cls)
        print(split, cls, "->", len(os.listdir(p)))

In [ ]:
import random
import cv2
import matplotlib.pyplot as plt

folder = os.path.join(RGB_CLEAN_DIR, "train", "Fight")
vid = random.choice(os.listdir(folder))
path = os.path.join(folder, vid)

cap = cv2.VideoCapture(path)
ok, frame = cap.read()
cap.release()

print("Video:", path, "Read OK:", ok)
frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
plt.imshow(frame_rgb)
plt.axis("off")
plt.show()

Dataset class

In [ ]:
import os, random
from glob import glob
import torch
from torch.utils.data import Dataset
from decord import VideoReader, cpu
import torchvision.transforms as T

class RWF2000VideoDataset(Dataset):
    def __init__(self, root=RGB_CLEAN_DIR, split="train", num_frames=8, size=112, mode="train"):
        self.root = root
        self.split = split
        self.num_frames = num_frames
        self.size = size
        self.mode = mode

        self.class_to_idx = {"NonFight": 0, "Fight": 1}
        self.samples = []

        for cls, label in self.class_to_idx.items():
            folder = os.path.join(root, split, cls)
            vids = sorted(glob(os.path.join(folder, "*.avi")))
            self.samples += [(v, label) for v in vids]

        if mode == "train":
            self.tf = T.Compose([
                T.Resize((size, size)),
                T.RandomHorizontalFlip(0.5),
                T.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
            ])
        else:
            self.tf = T.Compose([
                T.Resize((size, size)),
                T.Normalize([0.485,0.456,0.406], [0.229,0.224,0.225]),
            ])

    def __len__(self):
        return len(self.samples)

    def _sample_indices(self, n):
        T_ = self.num_frames
        if n >= T_:
            if self.mode == "train":
                start = random.randint(0, n - T_)
                return list(range(start, start + T_))
            else:
                return torch.linspace(0, n - 1, T_).long().tolist()
        else:
            return list(range(n)) + [n - 1] * (T_ - n)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        vr = VideoReader(path, ctx=cpu(0))
        n = len(vr)
        inds = self._sample_indices(n)

        frames = vr.get_batch(inds).asnumpy()
        frames = torch.from_numpy(frames).float() / 255.0
        frames = frames.permute(0, 3, 1, 2)

        out = []
        for t in range(frames.shape[0]):
            out.append(self.tf(frames[t]))
        frames = torch.stack(out, dim=0)

        clip = frames.permute(1, 0, 2, 3)
        return clip, torch.tensor(label)

Dataset test

In [ ]:
train_ds = RWF2000VideoDataset(root=RGB_CLEAN_DIR, split="train", mode="train")
val_ds = RWF2000VideoDataset(root=RGB_CLEAN_DIR, split="val", mode="val")

print("Train:", len(train_ds))
print("Val:", len(val_ds))

x, y = train_ds[0]
print("Clip shape:", x.shape)
print("Label:", y)

# RGB Training

In [ ]:
!rm -rf /content/rwf2000_clean
!rsync -av "/content/drive/MyDrive/violence-detection/data/rwf2000_clean/" /content/rwf2000_clean/

In [ ]:
import glob

print(len(glob.glob("/content/rwf2000_clean/train/Fight/*.avi")))
print(len(glob.glob("/content/rwf2000_clean/train/NonFight/*.avi")))
print(len(glob.glob("/content/rwf2000_clean/val/Fight/*.avi")))
print(len(glob.glob("/content/rwf2000_clean/val/NonFight/*.avi")))

Create RGB dataloaders

In [ ]:
from torch.utils.data import DataLoader
from glob import glob

# create datasets (choose your DATA_ROOT from Drive or /content)
train_ds = RWF2000VideoDataset(root=DATA_ROOT, split="train", num_frames=16, size=224, mode="train")
val_ds   = RWF2000VideoDataset(root=DATA_ROOT, split="val",   num_frames=16, size=224, mode="val")

train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=2, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=2, pin_memory=True, persistent_workers=True)

x, y = next(iter(train_loader))
print("x:", x.shape, "y:", y.shape)   # x should be (B, 3, 16, 224, 224)


Build RGB baseline model

In [ ]:
import torch
import torch.nn as nn
from torchvision.models.video import r3d_18

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = r3d_18(weights="DEFAULT")
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

for param in model.parameters():
    param.requires_grad = False

for param in model.fc.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(
    model.fc.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

Training Function

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
from tqdm.auto import tqdm
import torch

def run_one_epoch(model, loader, criterion, optimizer=None, device="cpu", train=True):
    model.train() if train else model.eval()

    total_loss = 0.0
    all_preds, all_labels = [], []

    pbar = tqdm(loader, desc=("Train" if train else "Val"), leave=False)

    for clips, labels in pbar:
        clips = clips.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        with torch.set_grad_enabled(train):
            outputs = model(clips)
            loss = criterion(outputs, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        pbar.set_postfix(loss=f"{loss.item():.4f}")

        total_loss += loss.item() * clips.size(0)
        preds = torch.argmax(outputs, dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    acc = accuracy_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)

    return avg_loss, acc, f1

Train RGB model

In [ ]:
import os

SAVE_DIR = "/content/drive/MyDrive/violence-detection/checkpoints/rgb"
os.makedirs(SAVE_DIR, exist_ok=True)

EPOCHS = 10

history = {
    "train_loss": [],
    "train_acc": [],
    "train_f1": [],
    "val_loss": [],
    "val_acc": [],
    "val_f1": [],
}

for epoch in range(1, EPOCHS+1):

    train_loss, train_acc, train_f1 = run_one_epoch(
        model, train_loader, criterion, optimizer, device, train=True
    )

    val_loss, val_acc, val_f1 = run_one_epoch(
        model, val_loader, criterion, optimizer=None, device=device, train=False
    )

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["train_f1"].append(train_f1)

    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)

    print(f"Epoch {epoch}")
    print(f"Train: loss={train_loss:.4f} acc={train_acc:.3f} f1={train_f1:.3f}")
    print(f"Val:   loss={val_loss:.4f} acc={val_acc:.3f} f1={val_f1:.3f}")
    print("-"*40)

    # 🔥 SAVE CHECKPOINT
    checkpoint_path = os.path.join(SAVE_DIR, f"r3d_epoch_{epoch}.pth")

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss
    }, checkpoint_path)

    print(f"Checkpoint saved at {checkpoint_path}")

In [ ]:
import json
import os

history_path = "/content/drive/MyDrive/violence-detection/results/rgb_training_history.json"

with open(history_path, "w") as f:
    json.dump(history, f)

print("Saved history to:", history_path)

Resume training from checkpoint (optional)

In [ ]:
import torch
import torch.nn as nn
from torchvision.models.video import r3d_18

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = r3d_18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 2)
model = model.to(device)

ckpt_path = "/content/drive/MyDrive/violence-detection/checkpoints/rgb/r3d_epoch_8.pth"
checkpoint = torch.load(ckpt_path, map_location=device)

model.load_state_dict(checkpoint["model_state_dict"])

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

start_epoch = checkpoint["epoch"] + 1
total_epochs = 20

In [ ]:
import os
import torch

SAVE_DIR = "/content/drive/MyDrive/violence-detection/checkpoints/rgb"
os.makedirs(SAVE_DIR, exist_ok=True)

for epoch in range(start_epoch, total_epochs + 1):

    train_loss, train_acc, train_f1 = run_one_epoch(
        model, train_loader, criterion, optimizer, device, train=True
    )

    val_loss, val_acc, val_f1 = run_one_epoch(
        model, val_loader, criterion, optimizer=None, device=device, train=False
    )

    print(f"Epoch {epoch}")
    print(f"Train: loss={train_loss:.4f} acc={train_acc:.3f} f1={train_f1:.3f}")
    print(f"Val:   loss={val_loss:.4f} acc={val_acc:.3f} f1={val_f1:.3f}")
    print("-" * 40)

    checkpoint_path = os.path.join(SAVE_DIR, f"r3d_epoch_{epoch}.pth")

    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss
    }, checkpoint_path)

    print(f"Checkpoint saved at {checkpoint_path}")